# Domain 1: Agentic Architecture & Orchestration
## TS 1.4: Implement Multi-Step Workflows with Enforcement and Handoff Patterns
**Certification Focus**: CCAR-F Domain 1 (27% Exam Weight) | **Scenario Alignment**: Scenario 1 (Customer Support Resolution Agent)

---

### 🎯 Key Architectural Principles
1. **Programmatic Prerequisite Gates**: Business policies (e.g., verifying customer identity before processing refunds) **must** be enforced via code state machines, not merely system prompt instructions.
2. **Prompts vs Code**: Prompts have a non-zero probability of non-compliance (hallucinations, prompt injections, ambiguity). Programmatic gates provide **100% deterministic compliance**.
3. **Structured Handoff Manifests**: Escalation to human supervisors requires compiling a structured summary (customer ID, issue summary, actions attempted, root cause, recommended action).
4. **Immediate Escalation Triggers**: Explicit user demand for a human, unhandled policy gaps, or repeated failure must immediately halt automated processing.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.4 Prerequisite Enforcement.")

### 🛡️ Step 2: Workflow State & Programmatic Prerequisite Gate
The `WorkflowState` class acts as the gatekeeper. Any attempt to invoke `process_refund` without a verified customer ID is rejected with a structured `permission` error.

In [ ]:
# Step 2: State Machine & Prerequisite Gate
class WorkflowState:
    """State machine that deterministically blocks unverified operations."""
    def __init__(self) -> None:
        self.verified_customer_id: Optional[str] = None
        self.verified_customer_name: Optional[str] = None
        self.escalated_to_human: bool = False
        self.escalation_summary: Optional[Dict[str, Any]] = None

    def verify_customer(self, customer_id: str, name: str) -> None:
        self.verified_customer_id = customer_id
        self.verified_customer_name = name

    def check_gate(self, tool_name: str, params: Dict[str, Any]) -> Tuple[bool, Optional[Dict[str, Any]]]:
        """Deterministic Gate: Blocks financial/sensitive actions until identity is verified."""
        if tool_name in ["process_refund", "lookup_order"] and not self.verified_customer_id:
            return False, {
                "isError": True,
                "errorCategory": "permission",
                "isRetryable": False,
                "message": (
                    f"Deterministic Gate Rejection: Tool '{tool_name}' requires verified customer ID. "
                    "Must call get_customer first."
                ),
            }
        return True, None

print(" WorkflowState gatekeeper initialized.")

### 🔧 Step 3: Tool Definitions & Escalation Manifest
We implement tools and the structured escalation function that compiles the handoff manifest.

In [ ]:
# Step 3: Tools & Escalation Logic
MOCK_CUSTOMERS = {
    "alice@example.com": {"id": "CUST-101", "name": "Alice Smith"},
}

def get_customer(email: str, state: WorkflowState) -> Dict[str, Any]:
    rec = MOCK_CUSTOMERS.get(email.strip().lower())
    if rec:
        state.verify_customer(rec["id"], rec["name"])
        return {"found": True, "customer_id": rec["id"], "name": rec["name"]}
    return {"found": False, "isError": True, "errorCategory": "validation", "isRetryable": False}

def process_refund(order_id: str, amount: float, state: WorkflowState) -> Dict[str, Any]:
    allowed, err = state.check_gate("process_refund", {"order_id": order_id, "amount": amount})
    if not allowed:
        return err
    return {
        "status": "processed",
        "order_id": order_id,
        "customer_id": state.verified_customer_id,
        "amount_refunded": amount,
    }

def escalate_to_human(reason: str, recommended_action: str, state: WorkflowState) -> Dict[str, Any]:
    """Generates a comprehensive structured handoff manifest for a human supervisor."""
    state.escalated_to_human = True
    state.escalation_summary = {
        "customer_id": state.verified_customer_id or "UNVERIFIED",
        "customer_name": state.verified_customer_name or "UNKNOWN",
        "root_cause": reason,
        "recommended_action": recommended_action,
        "requires_manual_approval": True,
    }
    return {"escalated": True, "summary": state.escalation_summary}

print(" Business tools and escalation manifest compiler ready.")

### 🧪 Step 4: Offline Simulation Walkthrough
We demonstrate:
1. An unverified attempt to process a refund is **blocked** by the gate.
2. The user identity is verified.
3. The refund is **approved** and processed.

In [ ]:
# Step 4: Interactive Simulation
state = WorkflowState()

# Attempt 1: Call refund BEFORE identity verification -> Must be BLOCKED
print("--- Attempt 1: Refund before verification ---")
res1 = process_refund("ORD-991", 85.0, state)
print("Gate Result:", res1)

# Step 2: Verify customer
print("\n--- Step 2: Customer verification ---")
verify_res = get_customer("alice@example.com", state)
print("Verification Result:", verify_res)

# Attempt 2: Call refund AFTER identity verification -> Must SUCCEED
print("\n--- Attempt 3: Refund after verification ---")
res2 = process_refund("ORD-991", 85.0, state)
print("Refund Result:", res2)

### 🔬 Step 5: Verification Assertions
We assert that:
1. Gate deterministically blocked the first attempt with `permission` category and `isRetryable: False`.
2. Customer verification updated state.
3. Gate permitted the refund once verified.

In [ ]:
# Step 5: Verification Checks
assert res1["isError"] is True
assert res1["errorCategory"] == "permission"
assert res1["isRetryable"] is False

assert state.verified_customer_id == "CUST-101"
assert res2["status"] == "processed"
assert res2["amount_refunded"] == 85.0

# Verify human handoff manifest
handoff = escalate_to_human("Return policy dispute", "Issue courtesy credit", state)
assert handoff["escalated"] is True
assert handoff["summary"]["customer_id"] == "CUST-101"
assert handoff["summary"]["root_cause"] == "Return policy dispute"

print(" All verification assertions passed! Programmatic gatekeeper confirmed.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Rely on **code gates** (state machines, pre-execution checks) for prerequisite compliance, never on prompt instructions alone.
- **Rule 2**: When escalating to humans, always compile a structured handoff manifest (customer ID, attempted actions, root cause, recommended action).
- **Rule 3**: Blocked prerequisite calls should return `errorCategory: permission` with `isRetryable: False`.